# [Optional] Jupyter Notebook 101

**Goal**: 15 minutes to get comfortable with the notebook environment so the rest of the workshop is about Bedrock, not the editor.

If you've used Jupyter before — run the cells and skip to [Lab 1: Prompts 101](./01-prompts-101.ipynb).

## Learning Objectives

By the end of this notebook, you'll have:

1. Selected the **`.venv (Python 3.11)`** environment as the kernel
2. Run a markdown cell vs a code cell — and seen the difference
3. Defined a variable in one cell, modified it in another — observed how state persists
4. Run a shell command from a cell with `!`
5. Made a one-line Bedrock call and read the response
6. Practiced restarting the kernel (menu and from code) when state gets weird

---

## 1. Markdown vs code

You're reading a **markdown cell** right now. It renders text. Click it once and you'll see the raw markdown source — click anywhere outside or press `Esc` and it renders again.

The next cell is a **code cell**. Run it with `Shift + Enter`.

In [ ]:
print("Hello from Jupyter!")

You should see `Hello from Jupyter!` appear directly below the cell.

<div class="alert alert-block alert-info">

**If you got `ModuleNotFoundError` or the cell did nothing**: you haven't selected the right kernel. Click **Select Kernel** in the top-right → **Python Environments...** → **`.venv (Python 3.11)`** (marked *Recommended*).

</div>

---

## 2. State persists across cells

Variables, imports, function definitions — everything you create stays in the kernel until you restart it. Run the next two cells in order:

In [ ]:
greeting = "Hi from cell 2"
number = 42

In [ ]:
# This cell reads the variables defined above
print(greeting)
print(f"The number is: {number}")

Now **overwrite** `number` and print it again. Run the next cell and watch the value change from `42` to `50`:

In [ ]:
# Reassign `number` — the kernel keeps the latest value you give it
number = 50
print(f"The number is now: {number}")

You just saw it: `number` was `42`, the reassignment cell pushed it to `50`, and that new value persists for every cell that runs afterward. **The kernel remembers everything until you restart it.**

This is powerful — but also where most beginner mistakes come from. Skip a cell, and the next one breaks. Edit a function but forget to re-run, and the old version is still active. Re-run a cell that increments a value, and it climbs every time.

---

## 3. Running shell commands with `!`

A line starting with `!` runs as a **shell command**, not Python — handy for installing packages or inspecting files without leaving the notebook. The workshop's dependencies are already installed, but this is how you'd add one:

In [ ]:
# `!` sends the line to the shell. Examples (read-only here so they're safe to run):
!python --version
!pip list | grep -i boto3

# To install a package you'd write:  !pip install <package>
# To run any terminal command:       !ls -la  /  !aws sts get-caller-identity

## 4. The two shortcuts that matter

| Shortcut | What it does |
|---|---|
| `Shift + Enter` | Run cell, move to next |
| `Cmd/Ctrl + Enter` | Run cell, stay |

That's it. Run the next cell to see the result of the last expression auto-print:

In [ ]:
# Note: no print() needed — the last expression in a cell auto-displays
2 + 2

## 5. Your first Bedrock call

Now the real check: can this notebook talk to Amazon Bedrock?

The next cell:
1. Imports `boto3` (AWS SDK for Python)
2. Creates a Bedrock runtime client
3. Calls the **Converse API** with one user message
4. Prints the response

If this works, your AWS credentials are wired up correctly and you're ready for the rest of the workshop.

In [ ]:
import boto3

# Bedrock runtime client. Region must match where the workshop infra was deployed.
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

# Workshop default: Sonnet 4.6 via the global CRIS profile
MODEL_ID = "global.anthropic.claude-sonnet-4-6"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [{"text": "Say 'hello, workshop' in exactly three words."}],
        }
    ],
    inferenceConfig={"maxTokens": 50, "temperature": 0},
)

# Pretty-print the keys we care about
print("Reply:    ", response["output"]["message"]["content"][0]["text"])
print("Input:    ", response["usage"]["inputTokens"], "tokens")
print("Output:   ", response["usage"]["outputTokens"], "tokens")
print("Latency:  ", response["metrics"]["latencyMs"], "ms")

## 6. Restarting the kernel

Sometimes state gets weird — you've edited a function and forgotten which cells use the old version, or you've run cells out of order. The fix is always the same: **restart the kernel**, then re-run from the top.

Three ways to do it:

1. **Menu**: **Kernel → Restart Kernel** (or press `Esc` then `0 0` — zero twice)
2. **Restart and run everything in one step**: **Kernel → Restart Kernel and Run All Cells**
3. **From code/terminal**: a kernel is just a process, so you can restart it programmatically — see the cell below.

After a restart, the variable indicators reset and the cell after next will fail because `greeting` no longer exists:

In [ ]:
# Restart the kernel from code (uncomment to try). The kernel is just a process —
# this asks it to shut down so Jupyter relaunches a fresh one. All variables are cleared.
#
# import IPython
# IPython.Application.instance().kernel.do_shutdown(restart=True)
#
# Equivalent from a terminal: `jupyter kernelspec list` to find kernels; in VS Code/Jupyter
# the "Restart Kernel" button does the same thing. After restarting, re-run from the top.
print("Tip: uncomment the two lines above to restart the kernel programmatically.")

In [ ]:
# This will fail with NameError after a kernel restart, until you re-run cell 2 above
print(greeting)

## 7. Interrupting a cell

If you've written an infinite loop or a network call is stuck, you don't need to close the browser. The next cell holds a commented-out infinite loop — uncomment it, run the cell, then stop it with the **interrupt (■ stop) button** on the cell's toolbar:

In [ ]:
# Demo (uncomment to try): this cell loops forever so you can practice interrupting it.
# Run it, then click the interrupt button (the ■ stop icon beside the cell, or in the
# notebook toolbar) to stop it. Kernel → Interrupt Kernel does the same thing.
#
# import time
# i = 0
# while True:
#     i += 1
#     if i % 1_000_000 == 0:
#         print(f"still running... iteration {i}")
#     time.sleep(0.5)
print("Uncomment the loop above, run the cell, then stop it with the ■ interrupt button beside the cell.")

While the cell is running you'll see `[*]` next to it. Click the **■ interrupt (stop) button** beside the cell — or **Kernel → Interrupt Kernel**. The cell stops with a `KeyboardInterrupt` — that's the kernel saying "I heard you, I stopped". The kernel itself is still alive; you can keep using it.

---

## You're ready

If you've run this whole notebook and the Bedrock call worked, you have everything you need to move on.

<div class="alert alert-block alert-success">

**Next**: [Lab 1: Prompts 101](./01-prompts-101.ipynb) — token counting, pricing, latency, the Converse API, and Bedrock Mantle.

</div>